# Tutorial: AI Engram TOFU Unlearning Audit Walkthrough

**대상:** AI Engram의 TOFU unlearning 결과를 처음 재현하거나 감사하는 연구자.

**선행 지식:** Python, PyTorch의 기본적인 model/forward 개념. 전체 실험에는 CUDA GPU와 Hugging Face 접근 권한이 필요합니다.

**학습 목표:**
- covariance 기반 engram 편집이 무엇을 빼는지 설명한다.
- 정적 forgetting과 실제 erasure를 구분한다.
- 공통 모듈로 기본 편집을 실행하고 전체 감사 배터리를 순서대로 재현한다.
- 저장된 결과에서 suppression-vs-erasure 결론과 한계를 읽는다.


## 전체 흐름

1. 환경과 저장된 결과 확인
2. AI Engram의 핵심 이론
3. TOFU 데이터와 answer-token masking
4. covariance 수집 → engram 계산 → 가중치 편집
5. 정적 forget/retain NLL 평가
6. relearning·novel/entity·SHAM·ICL 통제 실험
7. forget05·3B 교차검증
8. 결과 해석, 함정, 연습

> 기본 실행은 저장된 결과만 읽으므로 CPU에서 빠르게 끝납니다. GPU 실험은 `RUN_GPU = True`로 명시적으로 켭니다.


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "src" / "engram").exists(), "ai-engram 레포 안에서 실행하세요."

RESULTS = REPO_ROOT / "experiments" / "unlearning_audit" / "results"
RUN_GPU = False
print(f"repo={REPO_ROOT}\nresults={RESULTS}\nrun_gpu={RUN_GPU}")


## 1. 현재 결과를 먼저 확인하기

재실행 전에 이미 기록된 결과의 범위와 파일 무결성을 확인합니다. 이것이 재현 기준점입니다.


In [ ]:
result_files = sorted(RESULTS.glob("*.json"))
assert result_files, "저장된 audit JSON이 없습니다."
[(path.name, path.stat().st_size) for path in result_files]


## 2. 이론: covariance에서 engram 편집까지

레이어 입력 활성화 $x$에 대해 target 집합과 전체 기준 집합의 2차 통계를 모읍니다.

$$G_f = E_{x\sim forget}[xx^T], \qquad G_t = E_{x\sim total}[xx^T]$$

`EngramEditor`는 이 통계에서 forget 방향의 projection $P_l$을 레이어별로 계산하고, 가중치를 다음처럼 편집합니다.

$$W'_l = W_l - \alpha f_l P_l$$

- `count_ratio`: forget/total 표본 수로 $f_l$을 보정
- `weight_norm`: 레이어별 $\|P_l\|/\|W_l\|$로 편집 강도를 보정
- answer-token mask: 질문 형식이 아니라 답변 토큰에서만 통계를 수집

중요한 점: 출력 확률이 낮아졌다는 사실만으로 내부 정보가 **삭제(erasure)**됐다고 결론 내릴 수 없습니다. 빠른 relearning이나 동일 개념 방향의 probe transfer가 남으면 **억제(suppression)**일 수 있습니다.


## 3. 공통 모듈과 데이터 전처리

`experiments.unlearning_audit.tofu`가 tokenizer/model 로딩, answer-only labels, NLL, covariance 수집, 편집, embedding, fine-tuning을 한곳에서 제공합니다. 아래 GPU 셀은 `RUN_GPU=True`일 때만 모델과 데이터를 다운로드합니다.


In [ ]:
if RUN_GPU:
    sys.path.insert(0, str(REPO_ROOT))
    sys.path.insert(0, str(REPO_ROOT / "src"))
    from datasets import load_dataset
    from experiments.unlearning_audit.tofu import (
        BASE_ID, edit_model, load_model, load_tokenizer, mean_answer_nll,
    )

    device = "cuda"
    tokenizer = load_tokenizer()
    base = load_model(device=device).eval()
    full = list(load_dataset("locuslab/TOFU", "full")["train"])
    forget = list(load_dataset("locuslab/TOFU", "forget10_perturbed")["train"])
    retain = list(load_dataset("locuslab/TOFU", "retain_perturbed")["train"])
    print(BASE_ID, len(full), len(forget), len(retain))
else:
    print("GPU 단계 생략: RUN_GPU=True로 바꾸면 데이터와 모델을 로드합니다.")


## 4. 최소 편집과 정적 평가

공통 함수 하나가 forget/total covariance를 수집하고 adaptive-norm 편집을 적용합니다. forget NLL은 크게 증가하고 retain NLL은 비교적 유지되어야 합니다.


In [ ]:
if RUN_GPU:
    retain_eval = retain[:200]
    before = {
        "forget": mean_answer_nll(base, forget, tokenizer, device),
        "retain": mean_answer_nll(base, retain_eval, tokenizer, device),
    }
    edited = edit_model(base, forget, full, tokenizer, device)
    after = {
        "forget": mean_answer_nll(edited, forget, tokenizer, device),
        "retain": mean_answer_nll(edited, retain_eval, tokenizer, device),
    }
    print({"before": before, "after": after})
else:
    print("GPU 단계 생략. 저장된 ablation.json과 final report에서 기준 결과를 확인하세요.")


## 5. 전체 감사 배터리

각 파일은 한 질문만 담당하며 공통 runtime을 재사용합니다.

| 순서 | 모듈 | 질문 |
|---:|---|---|
| 1 | `relearn_attack` | 편집 모델이 never-knew gold보다 빨리 회복하는가? |
| 2 | `novel_control` | 회복이 일반적인 과가소성이 아니라 forget 지식에 특이적인가? |
| 3 | `entity_control` | entity familiarity를 통제해도 fact trace가 남는가? |
| 4 | `alpha_sweep` | 강한 편집이 erasure를 깊게 하는가, utility만 훼손하는가? |
| 5 | `sham_scr` | 동일 substrate의 가짜 편집으로 edit-hole 설명을 통제할 수 있는가? |
| 6 | `generation`, `length_match` | ICL/표면형식/문장 길이 confound를 통제하면 무엇이 남는가? |
| 7 | `crossval_forget05`, `crossval_3b` | split과 모델 크기를 바꿔도 방향이 반복되는가? |


In [ ]:
PIPELINE = [
    "relearn_attack", "novel_control", "entity_control", "alpha_sweep",
    "sham_scr", "generation", "length_match",
    "crossval_forget05", "crossval_3b",
]

def run_experiment(name: str) -> None:
    assert name in PIPELINE
    env = {"PYTHONPATH": str(REPO_ROOT / "src")}
    subprocess.run(
        [sys.executable, "-m", f"experiments.unlearning_audit.{name}"],
        cwd=REPO_ROOT,
        env={**__import__("os").environ, **env},
        check=True,
    )

if RUN_GPU:
    for experiment in PIPELINE:
        run_experiment(experiment)
else:
    print("실행 순서:", " -> ".join(PIPELINE))


## 6. 핵심 결과 읽기

점 추정 하나보다 통제 실험의 방향이 일관적인지 확인합니다. 아래 셀은 저장된 JSON에서 핵심 판정 입력만 압축해 읽습니다.


In [ ]:
def read_result(name: str):
    return json.loads((RESULTS / name).read_text())

novel = read_result("novel_control.json")
entity = read_result("entity_control.json")
x05 = read_result("xval_forget05.json")
x3b = read_result("xval_3b.json")

summary = {
    "forget-specific recovery R": novel.get("recovery_fraction_forget"),
    "edited-vs-entity-control gap": entity.get("verdict_gap_edited_vs_goldef"),
    "forget05 prereg": x05.get("prereg"),
    "3B prereg": x3b.get("prereg"),
}
summary


### 해석

- forget-specific relearning과 entity 통제는 단순한 과가소성/친숙도 설명보다 suppression 해석을 지지합니다.
- forget05와 3B에서도 dissociation 방향은 반복되지만 context-collapse 기준은 반복되지 않았습니다.
- 따라서 결론은 “AI Engram이 모든 정보를 영구 삭제한다”가 아니라, **이 설정의 정적 forgetting은 내부 erasure의 충분한 증거가 아니다**입니다.
- 외적 타당도는 TOFU와 Llama-3.2 계열 밖에서 추가 검증이 필요합니다.


## 흔한 실수와 확장

- **실수:** forget 집합만으로 total covariance를 대신함 → `count_ratio` calibration이 깨져 과도한 편집 가능.
- **실수:** 낮은 답변 확률을 곧바로 erasure로 해석함 → gold/novel/entity 통제와 relearning이 필요.
- **실수:** 1B의 동일 alpha를 3B에 그대로 해석함 → 실험에서 scale 간 dose 비이식성이 관찰됨.
- **선택 확장:** 다른 모델 계열과 실제 개인정보/저작권 데이터에서 동일한 사전등록 battery 반복.


## 연습

1. `alpha_sweep.json`에서 adaptive 조건만 골라 `(static_forget, retain, recovery_R)`를 비교하세요.
2. alpha가 커질수록 recovery가 0으로 가는지, retain 손상이 커지는지 먼저 예측하세요.


In [ ]:
alpha = read_result("alpha_sweep.json")
adaptive = [row for row in alpha["rows"] if row["scale"] == "adaptive"]
[(row["alpha"], row["static_forget"], row["retain"], row["recovery_R"]) for row in adaptive]


**정답 해설:** 강한 alpha가 정적 forget NLL과 collateral damage를 함께 키우더라도 recovery가 gold 수준($R\approx0$)으로 가지 않으면, erasure 깊이보다 suppression/utility trade-off로 해석하는 편이 안전합니다.

전체 표와 실험별 caveat는 `experiments/unlearning_audit/results/final_report_260711.html`을 참고하세요.
